In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.special import expit

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [3]:
N = CONFIG['n_sim']
rng = np.random.default_rng(SEED)

In [4]:
TAG = 'ar6_6'
GENERATED_FROM = 6

In [5]:
params = pd.read_csv(INPUT / TAG / 'parameters.csv', dtype={'stage':str})
panel = pd.read_csv(INPUT / 'data/panel.csv', index_col='dblp_id')
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)
sim = np.zeros((Y+1, N))

In [6]:
# empirical six-year prefixes; joint resampling retains their dependence
Q = panel.to_numpy(dtype=float)
valid = np.isfinite(Q[:,:6]).all(axis=1)
prefix_ids = rng.choice(np.flatnonzero(valid), N, replace=True)
sim[:6] = Q[prefix_ids,:6].T
pd.DataFrame({'sim_id':np.arange(N), 'prefix_scholar':panel.index[prefix_ids]}).to_csv(DEST / 'prefix_sources.csv', index=False)
p = params.iloc[0]
for t in range(5,Y):
    x = np.log1p(sim[t-5:t+1][::-1].T)
    active = rng.random(N) < expit(p.activity_intercept + x@params.activity_beta.to_numpy())
    magnitude = p.positive_intercept + x@params.positive_beta.to_numpy() + rng.normal(0,p.positive_sigma,N)
    sim[t+1,active] = np.exp(magnitude[active])

In [7]:
np.save(DEST / 'trajectories.npy', sim, allow_pickle=False)
pd.DataFrame([{'model':TAG, 'seed':SEED, 'n_sim':N, 'n_years':Y+1, 'generated_from_age':GENERATED_FROM,
               'orientation':'age x scholar', 'minimum':sim.min(), 'maximum':sim.max(),
               'finite':np.isfinite(sim).all()}]).to_csv(DEST / 'metadata.csv', index=False)
display(pd.read_csv(DEST / 'metadata.csv'))

,model,seed,n_sim,n_years,generated_from_age,orientation,minimum,maximum,finite
0,ar6_6,63,10000,21,6,age x scholar,0.0,136.11939,True
